# AI Powered Payroll anomaly Detection

This week task is to:
1. **Improve forecasting model by adding:**
   * Department-level predictions
   * overtime trend predictions
2. **Predicting**
   * Absenteesim risk
   * Payroll manipulation risk
3. **Model monitoring setup**
   * Drift
   * Accuracy drops
   * alert when performance drops

In [304]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

In [305]:
payroll_data = pd.read_csv("../data/refined_data.csv")
payroll_data

,employee_id,employee_name,department,designation,base_salary,joining_date,working_days,present_days,overtime_hours,gross_salary,...,salary_if_score_norm,deduction_if_score_norm,overtime_if_score_norm,salary_hybrid_score,deduction_hybrid_score,overtime_hybrid_score,risk_binary,system_alert_flag,rule_based_anomaly,year_month
0,1001,Aarav,IT,Developer,50000,2023-01-31,22,20,5,52500,...,0.502899,0.902217,0.796294,0.533866,0.296383,0.612978,0,1,1,2023-01
1,1002,Diya,HR,HR Executive,35000,2023-02-28,22,21,2,36000,...,0.904858,0.920234,0.985057,0.449555,0.683588,0.976850,0,1,1,2023-02
2,1003,Rohan,Finance,Accountant,40000,2023-03-31,22,19,0,40000,...,0.912878,0.599176,0.442468,0.600168,0.446420,1.595386,0,1,0,2023-03
3,1004,Sneha,IT,Developer,55000,2023-04-30,22,22,8,59000,...,0.663913,0.796936,0.677769,0.675220,0.354553,0.829139,0,1,1,2023-04
4,1005,Karan,Sales,Sales Exec,30000,2023-05-31,22,20,4,32000,...,0.597719,0.119110,0.732120,0.649752,0.551755,0.824958,0,1,0,2023-05
5,1006,Meera,Marketing,Marketing Exec,32000,2023-06-30,22,21,3,33500,...,1.000000,0.673016,1.000000,0.314154,0.676301,0.958898,0,1,1,2023-06
6,1007,Vikram,IT,Senior Developer,80000,2023-07-31,22,22,10,85000,...,0.686361,0.799446,0.676230,1.207174,0.964106,1.737460,1,1,1,2023-07
7,1008,Ananya,HR,HR Manager,60000,2023-08-31,22,20,1,60500,...,0.892205,0.646165,0.826709,0.444877,0.424389,0.643439,0,1,1,2023-08
8,1009,Rahul,Finance,Finance Manager,70000,2023-09-30,22,21,2,71000,...,0.790102,0.244937,0.694085,1.112980,0.558226,0.752551,0,1,1,2023-09
9,1010,Pooja,Sales,Sales Manager,65000,2023-10-31,22,19,6,68000,...,0.819782,0.202959,0.266669,1.233276,0.415943,0.883879,0,1,1,2023-10


In [306]:
df = payroll_data.copy()
df['year_month'] = pd.to_datetime(df['year_month'])
df = df.sort_values('year_month')
df.head()

,employee_id,employee_name,department,designation,base_salary,joining_date,working_days,present_days,overtime_hours,gross_salary,...,salary_if_score_norm,deduction_if_score_norm,overtime_if_score_norm,salary_hybrid_score,deduction_hybrid_score,overtime_hybrid_score,risk_binary,system_alert_flag,rule_based_anomaly,year_month
0,1001,Aarav,IT,Developer,50000,2023-01-31,22,20,5,52500,...,0.502899,0.902217,0.796294,0.533866,0.296383,0.612978,0,1,1,2023-01-01
1,1002,Diya,HR,HR Executive,35000,2023-02-28,22,21,2,36000,...,0.904858,0.920234,0.985057,0.449555,0.683588,0.976850,0,1,1,2023-02-01
2,1003,Rohan,Finance,Accountant,40000,2023-03-31,22,19,0,40000,...,0.912878,0.599176,0.442468,0.600168,0.446420,1.595386,0,1,0,2023-03-01
3,1004,Sneha,IT,Developer,55000,2023-04-30,22,22,8,59000,...,0.663913,0.796936,0.677769,0.675220,0.354553,0.829139,0,1,1,2023-04-01
4,1005,Karan,Sales,Sales Exec,30000,2023-05-31,22,20,4,32000,...,0.597719,0.119110,0.732120,0.649752,0.551755,0.824958,0,1,0,2023-05-01


In [307]:
df['attendance_ratio'] = df['present_days'] / df['working_days']
df['absent_days'] = df['working_days'] - df['present_days']
df['overtime_ratio'] = df['overtime_hours'] / df['working_days']
df['tenure_days'] = (df['year_month'] - pd.to_datetime(df['joining_date'])).dt.days
df['salary_growth'] = df.groupby('employee_id')['net_salary'].pct_change().fillna(0)
df['dept_avg_salary'] = df.groupby('department')['net_salary'].transform('mean')
df['dept_overtime_avg'] = df.groupby('department')['overtime_hours'].transform('mean')

In [308]:
df['payroll_logic_fail_flag'] = (
    (df['salary_diff_flag'] == 1) |
    (df['rule_based_anomaly'] == 1)
).astype(int)

payroll_logic_fail_rate = df['payroll_logic_fail_flag'].mean()
PAYROLL_LOGIC_FAILURE_ALERT = payroll_logic_fail_rate > 0.2

In [309]:
anomaly_rate = df['final_anomaly_flag'].mean()
MASS_ANOMALY_ALERT = anomaly_rate > 0.25

In [310]:
deduction_anomaly_rate = df['deduction_anomaly_flag'].mean()
DEDUCTION_SYSTEM_ALERT = deduction_anomaly_rate > 0.2

In [311]:
def calculate_psi(expected, actual, buckets=10):
    breakpoints = np.linspace(0,1,buckets+1)
    expected_perc = np.histogram(expected, bins=buckets)[0] / len(expected)
    actual_perc = np.histogram(actual, bins=buckets)[0] / len(actual)
    psi = np.sum((expected_perc - actual_perc) * np.log((expected_perc + 1e-5)/(actual_perc + 1e-5)))
    return psi

In [312]:
previous = df[df['year_month'] < df['year_month'].max()]
current = df[df['year_month'] == df['year_month'].max()]
psi_salary = calculate_psi(previous['net_salary'], current['net_salary'])
psi_overtime = calculate_psi(previous['overtime_hours'], current['overtime_hours'])

In [313]:
SALARY_DRIFT_ALERT = psi_salary > 0.25
OVERTIME_DRIFT_ALERT = psi_overtime > 0.25

In [314]:
SYSTEM_ALERT = any([
    PAYROLL_LOGIC_FAILURE_ALERT,
    MASS_ANOMALY_ALERT,
    DEDUCTION_SYSTEM_ALERT,
    SALARY_DRIFT_ALERT,
    OVERTIME_DRIFT_ALERT
])

In [315]:
print("------ SYSTEM ALERT ------")
print(SYSTEM_ALERT)

------ SYSTEM ALERT ------
True


In [316]:
print("------ SYSTEM HEALTH REPORT ------")

print("Payroll Logic Failure Rate:", payroll_logic_fail_rate)
print("Anomaly Rate:", anomaly_rate)
print("Deduction Anomaly Rate:", deduction_anomaly_rate)
print("Salary PSI:", psi_salary)
print("Overtime PSI:", psi_overtime)
print("PAYROLL_LOGIC_FAILURE_ALERT:", PAYROLL_LOGIC_FAILURE_ALERT)
print("MASS_ANOMALY_ALERT:", MASS_ANOMALY_ALERT)
print("DEDUCTION_SYSTEM_ALERT:", DEDUCTION_SYSTEM_ALERT)
print("SALARY_DRIFT_ALERT:", SALARY_DRIFT_ALERT)
print("OVERTIME_DRIFT_ALERT:", OVERTIME_DRIFT_ALERT)
print("SYSTEM_ALERT:", SYSTEM_ALERT)

------ SYSTEM HEALTH REPORT ------
Payroll Logic Failure Rate: 1.0
Anomaly Rate: 0.75
Deduction Anomaly Rate: 0.9
Salary PSI: 11.93801682926869
Overtime PSI: 10.465565564253858
PAYROLL_LOGIC_FAILURE_ALERT: True
MASS_ANOMALY_ALERT: True
DEDUCTION_SYSTEM_ALERT: True
SALARY_DRIFT_ALERT: True
OVERTIME_DRIFT_ALERT: True
SYSTEM_ALERT: True


# Forecasting model

## Payroll Forecasting model

In [317]:
monthly_payroll = df.groupby('year_month')['updated_net_salary'].sum().reset_index()
monthly_payroll['month_index'] = np.arange(len(monthly_payroll))

In [318]:
X = monthly_payroll[['month_index']]
y = monthly_payroll['updated_net_salary']

payroll_model = LinearRegression()
payroll_model.fit(X,y)

next_index = np.array([[monthly_payroll['month_index'].max()+1]])
monthly_payroll['predicted_payroll'] = payroll_model.predict(X)
predicted_month_payroll = payroll_model.predict(next_index)[0]

In [319]:
if SYSTEM_ALERT:
    adjustment_factor = (
        anomaly_rate * 0.15 +
        psi_salary * 0.1
    )
    alert_adjusted_payroll = predicted_month_payroll * (1 + adjustment_factor)
else:
    alert_adjusted_payroll = predicted_month_payroll

In [320]:
print("\n------ PAYROLL FORECAST REPORT ------")

print("Predicted Next Month Payroll:", round(predicted_month_payroll,2))
print("Alert Adjusted Payroll:", round(alert_adjusted_payroll,2))


------ PAYROLL FORECAST REPORT ------
Predicted Next Month Payroll: 52979.56
Alert Adjusted Payroll: 122186.86


## Department level Payroll forecasting

In [321]:
dept_monthly = df.groupby(['year_month','department'])['updated_net_salary'].sum().reset_index()
dept_monthly['month_index'] = dept_monthly.groupby('department').cumcount()

In [322]:
from sklearn.linear_model import LinearRegression

dept_predictions = {}
department_model = LinearRegression()
for dept in dept_monthly['department'].unique():
    temp = dept_monthly[dept_monthly['department'] == dept]
    X = temp[['month_index']]
    y = temp['updated_net_salary']
    
    department_model.fit(X, y)

    next_index = [[temp['month_index'].max() + 1]]
    normal_pred = department_model.predict(next_index)[0]
    dept_predictions[dept] = normal_pred

In [323]:
dept_predictions_with_drift = {}

for dept, pred in dept_predictions.items():

    if psi_salary > 0.25:
        drift_factor = 0.10
    elif psi_salary > 0.10:
        drift_factor = 0.05
    else:
        drift_factor = 0

    drift_adjusted_pred = pred * (1 + drift_factor)
    dept_predictions_with_drift[dept] = drift_adjusted_pred

In [324]:
dept_forecast_report = pd.DataFrame({
    "Department": list(dept_predictions.keys()),
    "Predicted_Payroll_No_Drift": list(dept_predictions.values()),
    "Predicted_Payroll_With_Drift": list(dept_predictions_with_drift.values())
})

print(dept_forecast_report)

  Department  Predicted_Payroll_No_Drift  Predicted_Payroll_With_Drift
0         IT                79316.818182                  87248.500000
1         HR                35242.045455                  38766.250000
2    Finance                57282.386364                  63010.625000
3      Sales                40361.363636                  44397.500000
4  Marketing                43369.696970                  47706.666667


In [325]:
dept_forecast_report['Drift_Status'] = "No Drift"

if psi_salary > 0.25:
    dept_forecast_report['Drift_Status'] = "Severe Drift"
elif psi_salary > 0.10:
    dept_forecast_report['Drift_Status'] = "Moderate Drift"

In [326]:
print("\n------ DEPARTMENT PAYROLL FORECAST ------")
dept_forecast_report


------ DEPARTMENT PAYROLL FORECAST ------


,Department,Predicted_Payroll_No_Drift,Predicted_Payroll_With_Drift,Drift_Status
0,IT,79316.818182,87248.500000,Severe Drift
1,HR,35242.045455,38766.250000,Severe Drift
2,Finance,57282.386364,63010.625000,Severe Drift
3,Sales,40361.363636,44397.500000,Severe Drift
4,Marketing,43369.696970,47706.666667,Severe Drift


## Overtime Trend Predictions

In [327]:
monthly_overtime = df.groupby('year_month')['overtime_hours'].sum().reset_index()
monthly_overtime['month_index'] = np.arange(len(monthly_overtime))

In [328]:
X = monthly_overtime[['month_index']]
y = monthly_overtime['overtime_hours']

overtime_model = LinearRegression()
overtime_model.fit(X,y)

next_index = [[monthly_overtime['month_index'].max()+1]]
predicted_overtime = overtime_model.predict(next_index)[0]

In [329]:
if OVERTIME_DRIFT_ALERT:
    predicted_overtime *= 1.1

In [330]:
print("\n------ OVERTIME TREND PREDICTION ------")

print("Predicted Overtime Hours Next Month:", round(predicted_overtime,2))


------ OVERTIME TREND PREDICTION ------
Predicted Overtime Hours Next Month: 5.07


# Employee Risk Model

## Absenteesim Risk

In [331]:
df['absenteeism_flag'] = (df['present_days'] < df['working_days']).astype(int)

In [332]:
employee_features = [
'LOP_days',
'paid_leaves',
'overtime_hours',
'attendance_ratio'
]

X = df[features]
y = df['absenteeism_flag']

In [333]:
absentee_risk_model = RandomForestClassifier()
absentee_risk_model.fit(X,y)

df['absenteeism_risk_score'] = absentee_risk_model.predict_proba(X)[:,1]

In [334]:
high_abs_risk = df[df['absenteeism_risk_score'] > 0.7]

In [335]:
print("Employees with High Absenteeism Risk\n")

print(high_abs_risk[['employee_id','employee_name','absenteeism_risk_score']])

Employees with High Absenteeism Risk

    employee_id employee_name  absenteeism_risk_score
0          1001         Aarav                    1.00
1          1002          Diya                    1.00
2          1003         Rohan                    1.00
4          1005         Karan                    1.00
5          1006         Meera                    1.00
7          1008        Ananya                    1.00
8          1009         Rahul                    1.00
9          1010         Pooja                    0.99
11         1012          Neha                    1.00
12         1013         Arjun                    1.00
13         1014         Kavya                    1.00
14         1015          Amit                    1.00
16         1017        Manish                    1.00
17         1018          Ritu                    1.00
18         1019        Nikhil                    1.00


## Salary Manipulation risk

In [336]:
risk_features = [
'salary_diff',
'salary_dev_pct',
'deduction_anomaly_score',
'salary_anomaly_score',
'rule_violation_score'
]

In [337]:
salary_risk_model = IsolationForest(contamination=0.1)
salary_risk_model.fit(df[risk_features])

df['salary_manipulation_risk'] = salary_risk_model.decision_function(df[risk_features])
df['salary_manipulation_flag'] = (salary_risk_model.predict(df[risk_features]) == -1).astype(int)

In [338]:
salary_risk_employees = df[df['salary_manipulation_flag']==1]

In [339]:
total_high_risk = len(salary_risk_employees)
high_risk_list = salary_risk_employees[['employee_id','employee_name','department']]

In [340]:
print("\n------ EMPLOYEE RISK REPORT ------")

print("Total High Risk Employees:", total_high_risk)
print("\nList of High Risk Employees:")
print(high_risk_list)


------ EMPLOYEE RISK REPORT ------
Total High Risk Employees: 2

List of High Risk Employees:
    employee_id employee_name department
4          1005         Karan      Sales
10         1011     Siddharth         IT


# Model Monitoring model

In [341]:
drift_report = pd.DataFrame({"metric":["salary_psi","overtime_psi"], "value":[psi_salary,psi_overtime]})

In [342]:
actual = monthly_payroll['updated_net_salary'].iloc[-1]
mape = mean_absolute_percentage_error([actual],[predicted_payroll])

In [343]:
ACCURACY_DECAY_ALERT = mape > 0.15

In [344]:
print("\n------ MODEL PERFORMANCE ------")

print("MAPE:", round(mape,3))
print("ACCURACY_DECAY_ALERT:", ACCURACY_DECAY_ALERT)


------ MODEL PERFORMANCE ------
MAPE: 0.292
ACCURACY_DECAY_ALERT: True


In [345]:
report = {
"System Alert": SYSTEM_ALERT,
"Payroll Logic Failure": PAYROLL_LOGIC_FAILURE_ALERT,
"Mass Anomaly": MASS_ANOMALY_ALERT,
"Salary Drift": SALARY_DRIFT_ALERT,
"Predicted Payroll": predicted_payroll,
"Alert Adjusted Payroll": alert_adjusted_payroll,
"Predicted Overtime": predicted_overtime,
"High Risk Employees": total_high_risk
}

print(report)

{'System Alert': True, 'Payroll Logic Failure': np.True_, 'Mass Anomaly': np.True_, 'Salary Drift': np.True_, 'Predicted Payroll': np.float64(52979.56339712918), 'Alert Adjusted Payroll': np.float64(122186.8562235298), 'Predicted Overtime': np.float64(5.0657894736842115), 'High Risk Employees': 2}


In [346]:
print("\n================ FINAL SUMMARY ================")

summary = {
"System Alert": SYSTEM_ALERT,
"Predicted Payroll": round(predicted_payroll,2),
"Alert Adjusted Payroll": round(alert_adjusted_payroll,2),
"Predicted Overtime": round(predicted_overtime,2),
"Total High Risk Employees": total_high_risk,
"Payroll Logic Failure Alert": PAYROLL_LOGIC_FAILURE_ALERT,
"Mass Anomaly Alert": MASS_ANOMALY_ALERT,
"Salary Drift Alert": SALARY_DRIFT_ALERT,
"Overtime Drift Alert": OVERTIME_DRIFT_ALERT,
"Accuracy Decay Alert": ACCURACY_DECAY_ALERT
}

for k,v in summary.items():
    print(k,":",v)


================ FINAL SUMMARY ================
System Alert : True
Predicted Payroll : 52979.56
Alert Adjusted Payroll : 122186.86
Predicted Overtime : 5.07
Total High Risk Employees : 2
Payroll Logic Failure Alert : True
Mass Anomaly Alert : True
Salary Drift Alert : True
Overtime Drift Alert : True
Accuracy Decay Alert : True


In [347]:
from sklearn.metrics import mean_absolute_percentage_error

In [348]:
def calculate_model_performance(actual, predicted):

    mape = mean_absolute_percentage_error(actual, predicted) * 100
    
    if mape < 10:
        performance = "EXCELLENT"
    elif mape < 15:
        performance = "ACCEPTABLE"
    else:
        performance = "POOR"
    
    return mape, performance

In [349]:
def detect_accuracy_decay(previous_mape, current_mape):

    decay = current_mape - previous_mape
    
    if decay > 5:
        alert = "FORECAST_ACCURACY_DECAY_ALERT"
    else:
        alert = "NO_ALERT"
    
    return decay, alert

In [350]:
def calculate_psi(expected, actual, buckets=10):

    expected_percents, bins = np.histogram(expected, bins=buckets)
    actual_percents, _ = np.histogram(actual, bins=bins)

    expected_percents = expected_percents / len(expected)
    actual_percents = actual_percents / len(actual)

    psi_values = (actual_percents - expected_percents) * np.log(
        (actual_percents + 1e-5) / (expected_percents + 1e-5)
    )

    psi = np.sum(psi_values)

    return psi

In [351]:
def interpret_drift(psi):

    if psi < 0.1:
        drift_status = "NO_DRIFT"
        alert = "NO_ALERT"

    elif psi < 0.25:
        drift_status = "MODERATE_DRIFT"
        alert = "DATA_DRIFT_WARNING"

    else:
        drift_status = "SIGNIFICANT_DRIFT"
        alert = "PAYROLL_DATA_DRIFT_ALERT"

    return drift_status, alert

In [352]:
def detect_drift_features(train_df, current_df, features):

    drift_report = {}

    for col in features:

        train_mean = train_df[col].mean()
        current_mean = current_df[col].mean()

        change_pct = abs((current_mean - train_mean) / (train_mean + 1e-5)) * 100

        drift_report[col] = round(change_pct,2)

    drift_report = dict(sorted(drift_report.items(), key=lambda x: x[1], reverse=True))

    return drift_report

In [353]:
def determine_root_cause(drift_report):

    major_feature = list(drift_report.keys())[0]
    change = drift_report[major_feature]

    if change > 30:
        cause = f"Major shift detected in {major_feature}"
    elif change > 15:
        cause = f"Moderate shift detected in {major_feature}"
    else:
        cause = "Minor feature distribution change"

    return cause

In [354]:
def generate_monitoring_report(
        df,
        previous_mape,
        actual_payroll,
        predicted_payroll):

    previous = df[df['year_month'] < df['year_month'].max()]
    current = df[df['year_month'] == df['year_month'].max()]

    current_mape, performance = calculate_model_performance(
        actual_payroll,
        predicted_payroll
    )

    decay, accuracy_alert = detect_accuracy_decay(
        previous_mape,
        current_mape
    )

    psi = calculate_psi(
        previous['updated_net_salary'],
        current['updated_net_salary']
    )

    drift_status, drift_alert = interpret_drift(psi)

    features = [
        "overtime_hours",
        "salary_dev_pct",
        "salary_diff",
        "present_days",
        "LOP_days"
    ]

    drift_report = detect_drift_features(previous, current, features)

    cause = determine_root_cause(drift_report)

    alerts = []

    if accuracy_alert != "NO_ALERT":
        alerts.append(accuracy_alert)

    if drift_alert != "NO_ALERT":
        alerts.append(drift_alert)

    if accuracy_alert != "NO_ALERT" and drift_alert != "NO_ALERT":
        alerts.append("CRITICAL_MODEL_HEALTH_ALERT")

    report = {
        "previous_mape": round(previous_mape,2),
        "current_mape": round(current_mape,2),
        "accuracy_decay": round(decay,2),
        "performance_status": performance,
        "psi_score": round(psi,3),
        "drift_status": drift_status,
        "root_cause": cause,
        "top_drift_features": list(drift_report.items())[:5],
        "alerts_generated": alerts
    }

    return report

In [355]:
monitoring_report = generate_monitoring_report(
    df=df,
    previous_mape=8.5,
    actual_payroll=monthly_payroll['updated_net_salary'],
    predicted_payroll=monthly_payroll['predicted_payroll']
)

for key, value in monitoring_report.items():
    print(f"{key} : {value}")

previous_mape : 8.5
current_mape : 37.79
accuracy_decay : 29.29
performance_status : POOR
psi_score : 21.191
drift_status : SIGNIFICANT_DRIFT
root_cause : Major shift detected in salary_dev_pct
top_drift_features : [('salary_dev_pct', np.float64(1970.42)), ('salary_diff', np.float64(226.75)), ('overtime_hours', np.float64(116.46)), ('LOP_days', np.float64(96.55)), ('present_days', np.float64(7.18))]
alerts_generated : ['FORECAST_ACCURACY_DECAY_ALERT', 'PAYROLL_DATA_DRIFT_ALERT', 'CRITICAL_MODEL_HEALTH_ALERT']


# Saving the models

In [356]:
import joblib
import os

os.makedirs("../saved_models", exist_ok=True)

In [357]:
joblib.dump(payroll_model, "../saved_models/payroll_forecast_model.pkl")

['../saved_models/payroll_forecast_model.pkl']

In [358]:
joblib.dump(department_model, "../saved_models/department_forecast_model.pkl")

['../saved_models/department_forecast_model.pkl']

In [359]:
joblib.dump(overtime_model, "../saved_models/overtime_forecast_model.pkl")

['../saved_models/overtime_forecast_model.pkl']

In [360]:
joblib.dump(absentee_risk_model, "../saved_models/absenteeism_risk_model.pkl")

['../saved_models/absenteeism_risk_model.pkl']

In [361]:
joblib.dump(salary_risk_model, "../saved_models/salary_risk_model.pkl")

['../saved_models/salary_risk_model.pkl']

In [362]:
joblib.dump(employee_features, "../saved_models/employee_features.pkl")

['../saved_models/employee_features.pkl']

In [363]:
joblib.dump(risk_features, "../saved_models/risk_features.pkl")

['../saved_models/risk_features.pkl']

In [365]:
forecast_features = ['month_index']
joblib.dump(forecast_features, "../saved_models/forecast_features.pkl")

['../saved_models/forecast_features.pkl']